# Notebook 06 — Linear SVM

Algoritmo: Linear SVM. Escalado: sí. n_iter = 15, CV=5, scoring='roc_auc'.

Observaciones: SVM lineal (RBF inviable con 38k samples). CalibratedClassifierCV para predict_proba.

## 1. Setup

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

RAIZ = Path('..').resolve()
sys.path.insert(0, str(RAIZ))

from src.preprocesamiento import preparar, RANDOM_STATE
from src.entrenamiento  import entrenar_y_evaluar

DATA = RAIZ / 'data'

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from scipy.stats import loguniform

## 2. Cargar y preparar datos

In [2]:
datos = preparar(DATA, aplicar_escalado=True)
print(f'Train: {datos.n_train} | Test: {datos.n_test} | Features: {len(datos.cols_features)}')
print(f'scale_pos_weight atraso: {datos.scale_pos_weight["tuvo_atraso"]:.4f}')


Train: 38664 | Test: 9667 | Features: 152
scale_pos_weight atraso: 0.6534


## 3. Clasificador + grid

In [3]:
clf = CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=RANDOM_STATE, dual='auto'), method='sigmoid', cv=3)

param_distributions = {
        'estimator__C': loguniform(0.01, 10),
        'estimator__class_weight': ['balanced', None],
    }

## 4. RandomizedSearchCV (15 iters × CV 5)

In [4]:
resultado = entrenar_y_evaluar(
    nombre='svm',
    estimador=clf,
    param_distributions=param_distributions,
    datos=datos,
    data_dir=DATA,
    n_iter=15,
    target='tuvo_atraso',
)
resultado


=== SVM ===
CV AUC:          0.8397
Test AUC:        0.8381
Test F1:         0.8144
Test Precision:  0.7705
Test Recall:     0.8635
Test Accuracy:   0.7619
Tiempo search:   222.67 s
Best params:     {'estimator__C': 1.3311216080736887, 'estimator__class_weight': None}


{'nombre': 'svm',
 'target': 'tuvo_atraso',
 'n_iter': 15,
 'cv_folds': 5,
 'tiempo_search_seg': 222.67,
 'best_params': {'estimator__C': 1.3311216080736887,
  'estimator__class_weight': None},
 'best_cv_auc': 0.8397,
 'AUC': 0.8381,
 'F1': 0.8144,
 'Precision': 0.7705,
 'Recall': 0.8635,
 'Accuracy': 0.7619}